# Inner-solver benchmark

Compare the inner relaxation solvers (SGD, Adam, nonlinear CG, L-BFGS) on the
mitotic-division pipeline: a 4-cell tissue taken through 25 consecutive divisions.
Each solver runs the **same** divisions (same cells, same axes) so only the
relaxation differs. We report wall-clock time, the real per-relaxation iteration
count (early stopping cuts most relaxations short of `max_opt_iter`), the cost per
iteration, the final mesh and its energy.

Self-contained: run top to bottom (Restart & Run All).


In [ ]:
import os, sys, math, sysconfig, time, importlib
# Make the mitotic_spindle modules importable without a hard-coded path.
_here = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
if _here not in sys.path:
    sys.path.insert(0, _here)
# jax 0.4.35 crashes auto-detecting the CUDA path (cuda_nvcc.__file__ is None), so
# CUDA_ROOT must be set BEFORE the first `import jax`.
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('JAX_ENABLE_X64', 'true')
_nvcc = os.path.join(sysconfig.get_paths()['purelib'], 'nvidia', 'cuda_nvcc')
if os.path.isdir(_nvcc):
    os.environ.setdefault('CUDA_ROOT', _nvcc)

import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from vertax import BoundedMesh, nonlinear_cg

# Reload local modules in dependency order (dependents last) so a change to the
# energy is always picked up by run_n_divisions.
import mesh_utils;       importlib.reload(mesh_utils)
import energy_adaptive as _ea_mod; importlib.reload(_ea_mod)
import cell_division;    importlib.reload(cell_division)
import spindle_model;    importlib.reload(spindle_model)
import visualization;    importlib.reload(visualization)
import spindle_analysis; importlib.reload(spindle_analysis)

from energy_adaptive import energy_adaptive
from spindle_analysis import run_n_divisions
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
print(jax.devices())


## Setup

Shared mesh builder, energy metrics, and the `run_solver_benchmark` driver. The
knobs `BENCH_N_CELLS`, `BENCH_N_DIV`, tolerance and patience apply to every solver.


In [ ]:
# Benchmark settings: a small 4-cell tissue (minimum for a Voronoi mesh) taken
# through 25 consecutive divisions -> 29 cells. Each solver gets its OWN iteration
# budget and early-stopping criterion, passed explicitly in its cell below.
BENCH_N_CELLS    = 4
BENCH_N_DIV      = 25
BENCH_TARGET_AREA = 0.6
# Default early-stopping criterion (per solver cell may override).
BENCH_TOLERANCE  = 1e-4
BENCH_PATIENCE   = 5


def mesh_metrics(m):
    """Energy decomposition + area/tension stats of a relaxed mesh, computed with
    the SAME energy that is minimised (energy_adaptive). Returns a dict with the
    global energy, its area vs tension parts, and mean area / mean tension."""
    vt = jnp.asarray(m.vertices); at = jnp.asarray(m.angles)
    he = jnp.asarray(m.edges);    fc = jnp.asarray(m.faces)
    vp = jnp.asarray(m.vertices_params)
    ep = jnp.asarray(m.edges_params); fp = jnp.asarray(m.faces_params)

    E_total = float(energy_adaptive(vt, at, he, fc, None, None, None, vp, ep, fp))

    # Area part: K_AREAS * sum (area - target)^2 (target = fp[0]); tension part is
    # the remainder of the total (inner + surface edge energy).
    areas = m.get_area(jnp.arange(m.nb_faces))
    E_area = float(_ea_mod.K_AREAS * jnp.sum((areas - fp[0]) ** 2))
    E_tension = E_total - E_area
    tensions = jax.nn.sigmoid(ep) + 1
    return dict(E=E_total, E_area=E_area, E_tension=E_tension,
                mean_area=float(jnp.mean(areas)), total_area=float(jnp.sum(areas)),
                mean_tension=float(jnp.mean(tensions)))


def make_bench_mesh():
    """Fresh 4-cell bounded mesh with the same tension/area conventions used
    throughout this notebook (so every solver starts from an identical mesh)."""
    w = math.sqrt(BENCH_N_CELLS)
    m = BoundedMesh.from_random_seeds(nb_seeds=BENCH_N_CELLS, width=w, height=w, random_key=2)
    m.vertices_params = jnp.asarray([0.0 for _ in range(m.nb_vertices)])
    key = jax.random.PRNGKey(3)
    m.edges_params = jax.random.uniform(key, (m.nb_edges,)) * 8 - 4  # raw; tension = sigmoid+1
    m.faces_params = jnp.asarray([BENCH_TARGET_AREA for _ in range(m.nb_faces)])
    return m


def run_solver_benchmark(name, solver, max_iter,
                         tolerance=BENCH_TOLERANCE, patience=BENCH_PATIENCE):
    """Run the full 25-division sequence with `solver` as inner solver, `max_iter`
    inner iterations per relaxation, and the given early-stopping (tolerance,
    patience). Returns (elapsed_seconds, initial_mesh, final_mesh, params_dict).
    cell_rng is re-seeded to rng(0) so every solver divides the same cells along
    the same axes."""
    m0 = make_bench_mesh()
    m0_snapshot = BoundedMesh.copy_mesh(m0)
    iters_per_relax = []   # actual iterations of each per-division relaxation
    t0 = time.perf_counter()
    m_final, _ = run_n_divisions(
        m0, n_divisions=BENCH_N_DIV,
        cell_rng=np.random.default_rng(0),
        inner_solver=solver,
        max_opt_iter=max_iter,
        tolerance=tolerance,
        patience=patience,
        iters_out=iters_per_relax,
    )
    elapsed = time.perf_counter() - t0
    # Early stopping (tolerance + patience) means each relaxation usually runs
    # fewer than max_iter iterations; record the real per-relaxation counts.
    total_iters = int(sum(iters_per_relax))
    mean_iters  = total_iters / max(len(iters_per_relax), 1)
    params = dict(max_iter=max_iter, tolerance=tolerance, patience=patience,
                  iters_per_relax=iters_per_relax, total_iters=total_iters,
                  mean_iters=mean_iters)
    met = mesh_metrics(m_final)
    print(f"[{name}] {elapsed:.2f} s, {total_iters} iters total "
          f"({mean_iters:.0f}/relax, cap {max_iter}), tol={tolerance:g}, "
          f"patience={patience}  ->  {m_final.nb_faces} cells  "
          f"|  E={met['E']:.3f} (area {met['E_area']:.3f} + tension {met['E_tension']:.3f})")
    return elapsed, m0_snapshot, m_final, params


# Collected per-solver results: name -> (elapsed_s, initial_mesh, final_mesh, params).
bench_results = {}

In [ ]:
# --- SGD -------------------------------------------------------------------------
# Per-solver iteration budget + early-stopping criterion (edit freely).
SGD_MAX_ITER  = 2000
SGD_TOLERANCE = 1e-4
SGD_PATIENCE  = 5
bench_results["SGD"] = run_solver_benchmark(
    "SGD", optax.sgd(learning_rate=1e-3), SGD_MAX_ITER,
    tolerance=SGD_TOLERANCE, patience=SGD_PATIENCE)
sgd_time = bench_results["SGD"][0]
print(f"SGD total time: {sgd_time:.2f} s")

In [ ]:
# --- Adam ------------------------------------------------------------------------
ADAM_MAX_ITER  = 2000
ADAM_TOLERANCE = 1e-4
ADAM_PATIENCE  = 5
bench_results["Adam"] = run_solver_benchmark(
    "Adam", optax.adam(learning_rate=1e-3), ADAM_MAX_ITER,
    tolerance=ADAM_TOLERANCE, patience=ADAM_PATIENCE)
adam_time = bench_results["Adam"][0]
print(f"Adam total time: {adam_time:.2f} s")

In [ ]:
# --- CG (nonlinear conjugate gradient) -------------------------------------------
# Line-search solvers usually need far fewer iterations than plain SGD/Adam.
CG_MAX_ITER  = 2000
CG_TOLERANCE = 1e-4
CG_PATIENCE  = 5
bench_results["CG"] = run_solver_benchmark(
    "CG", nonlinear_cg(restart_every=10), CG_MAX_ITER,
    tolerance=CG_TOLERANCE, patience=CG_PATIENCE)
cg_time = bench_results["CG"][0]
print(f"CG total time: {cg_time:.2f} s")

In [ ]:
# --- L-BFGS (requires optax >= 0.2.5; see note above) ----------------------------
LBFGS_MAX_ITER  = 2000
LBFGS_TOLERANCE = 1e-4
LBFGS_PATIENCE  = 5
bench_results["L-BFGS"] = run_solver_benchmark(
    "L-BFGS", optax.lbfgs(), LBFGS_MAX_ITER,
    tolerance=LBFGS_TOLERANCE, patience=LBFGS_PATIENCE)
lbfgs_time = bench_results["L-BFGS"][0]
print(f"L-BFGS total time: {lbfgs_time:.2f} s")

In [ ]:
# Initial mesh + the four final meshes (one per solver), side by side. The initial
# 4-cell mesh is shared by all solvers, so it is shown once; the four final meshes
# after 25 divisions sit next to it for a direct visual comparison of how each
# solver relaxed the tissue. Edge colour = tension, cell colour = area.
from visualization import _draw_mesh_on_ax, _add_cell_labels, AREA_SCALE, TENSION_SCALE
import matplotlib
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

sm_tension = ScalarMappable(Normalize(*TENSION_SCALE), matplotlib.colormaps["coolwarm"])
sm_area    = ScalarMappable(Normalize(*AREA_SCALE),    matplotlib.colormaps["cividis"])


def _fit_axes_to_mesh(ax, mesh, margin=0.3):
    """Fit the axes to the actual vertex extent (+ margin): after 25 divisions the
    mesh grows past its initial width/height, so the default limits would clip it."""
    v = np.asarray(mesh.vertices)
    xmin, ymin = v.min(axis=0); xmax, ymax = v.max(axis=0)
    ax.set_xlim(xmin - margin, xmax + margin)
    ax.set_ylim(ymin - margin, ymax + margin)


# Stable solver order regardless of dict insertion.
solver_order = [n for n in ["SGD", "Adam", "CG", "L-BFGS"] if n in bench_results]

# The initial mesh is the same for every solver — take it from the first result.
m_init = bench_results[solver_order[0]][1]
mi = mesh_metrics(m_init)

ncols = 1 + len(solver_order)          # initial + one final per solver
fig, axes = plt.subplots(1, ncols, figsize=(5.2 * ncols, 5.5))
axes = np.atleast_1d(axes)
fig.suptitle(f"Inner-solver benchmark — initial 4-cell mesh and the final meshes "
             f"after {BENCH_N_DIV} divisions  (edge = tension, cell = area)", fontsize=13)

# Panel 0: the shared initial mesh.
_draw_mesh_on_ax(m_init, axes[0], AREA_SCALE, TENSION_SCALE)
_fit_axes_to_mesh(axes[0], m_init)
_add_cell_labels(axes[0], m_init)
axes[0].set_title(f"initial ({m_init.nb_faces} cells)\nE={mi['E']:.2f}", fontsize=10)
axes[0].set_xticks([]); axes[0].set_yticks([])

# Panels 1..N: the final mesh of each solver.
for col, name in enumerate(solver_order, start=1):
    elapsed, _, m_final, params = bench_results[name]
    mf = mesh_metrics(m_final)
    ax = axes[col]
    _draw_mesh_on_ax(m_final, ax, AREA_SCALE, TENSION_SCALE)
    _fit_axes_to_mesh(ax, m_final)
    _add_cell_labels(ax, m_final)
    ax.set_title(f"{name} — final ({m_final.nb_faces} cells, {elapsed:.1f} s)\n"
                 f"E={mf['E']:.2f} (area {mf['E_area']:.2f} + tension {mf['E_tension']:.2f})",
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

# Single shared colorbar pair to the right of the whole row.
cbt = fig.colorbar(sm_tension, ax=list(axes), fraction=0.012, pad=0.03, aspect=15)
cba = fig.colorbar(sm_area,    ax=list(axes), fraction=0.012, pad=0.07, aspect=15)
cbt.set_label("tension"); cba.set_label("Area of cell")
plt.show()

In [ ]:
# Timing + quality summary — ranking from fastest to slowest over the 25-division run.
# Reports each solver's settings (iterations, tolerance, patience), wall-clock time,
# and the final mesh's global energy: fastest is not always lowest-energy.
ranked = sorted(bench_results.items(), key=lambda kv: kv[1][0])
fastest_name, (fastest_t, _, _, _) = ranked[0]

print(f"Inner-solver benchmark — 4-cell mesh, {BENCH_N_DIV} consecutive divisions")
print("-" * 113)
print(f"{'solver':8}  {'mean it':>7}  {'cap':>5}  {'tot it':>7}  {'ms/it':>7}  "
      f"{'tol':>7}  {'patience':>8}  {'time (s)':>9}  {'vs fast':>8}  {'E_final':>9}")
for name, (elapsed, _, m_final, params) in ranked:
    mf = mesh_metrics(m_final)
    # mean it = real iterations per relaxation (early-stopped, usually < cap).
    ms_per_it = elapsed / max(params['total_iters'], 1) * 1e3  # wall time per real iteration
    print(f"{name:8}  {params['mean_iters']:>7.0f}  {params['max_iter']:>5}  "
          f"{params['total_iters']:>7}  {ms_per_it:>7.1f}  {params['tolerance']:>7g}  "
          f"{params['patience']:>8}  {elapsed:>9.2f}  {elapsed / fastest_t:>7.2f}x  "
          f"{mf['E']:>9.2f}")
print("-" * 113)
print(f"Fastest: {fastest_name} ({fastest_t:.2f} s)")

# Three-panel summary: wall-clock time, cost per iteration, final energy.
# The cost-per-iteration panel explains why fewer iterations (CG/L-BFGS) need
# not mean less time: their line search makes each iteration far more expensive.
names = [n for n, _ in ranked]
times = [v[0] for _, v in ranked]
ms_iter = [v[0] / max(v[3]["total_iters"], 1) * 1e3 for _, v in ranked]
energies = [mesh_metrics(v[2])['E'] for _, v in ranked]
colors = ["#2a9d8f", "#457b9d", "#e9c46a", "#e76f51"][:len(names)]

fig, (ax_t, ax_c, ax_e) = plt.subplots(1, 3, figsize=(18, 4.5))
fig.suptitle(f"Inner-solver benchmark — {BENCH_N_DIV} divisions on a 4-cell mesh", fontsize=13)

bars_t = ax_t.bar(names, times, color=colors)
ax_t.set_ylabel("wall-clock time (s)")
ax_t.set_title("Total speed (lower is faster)")
for b, t in zip(bars_t, times):
    ax_t.text(b.get_x() + b.get_width() / 2, t, f"{t:.1f}s", ha="center", va="bottom", fontsize=9)

bars_c = ax_c.bar(names, ms_iter, color=colors)
ax_c.set_ylabel("wall time per iteration (ms)")
ax_c.set_title("Cost per iteration (line search makes CG/L-BFGS dear)")
for b, m in zip(bars_c, ms_iter):
    ax_c.text(b.get_x() + b.get_width() / 2, m, f"{m:.0f}", ha="center", va="bottom", fontsize=9)

bars_e = ax_e.bar(names, energies, color=colors)
ax_e.set_ylabel("final global energy  E")
ax_e.set_title("Solution quality (lower energy = better relaxation)")
for b, e in zip(bars_e, energies):
    ax_e.text(b.get_x() + b.get_width() / 2, e, f"{e:.1f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()